# 02 - Data Understanding

## Tujuan
Memahami struktur, kelengkapan, dan kualitas data sebelum diproses lebih lanjut,
serta mendokumentasikan dokumen yang dibuang beserta alasannya.

## Asal (Input)
`data/raw/thesis_scraped.csv` - 312 dokumen hasil Notebook 01

## Keluaran (Output)
- `data/processed/thesis_for_pipeline.csv` - 309 dokumen (filter awal)
- `docs/data_lineage_by_year.csv` - jejak jumlah dokumen per tahap
- `docs/excluded_documents.csv` - dokumen yang dibuang + alasannya

## Pertanyaan yang Dijawab
- Berapa dokumen yang punya field lengkap?
- Dokumen mana yang terduplikasi judulnya?
- Berapa dokumen yang lolos ke tahap berikutnya?

In [1]:
import sys, io as _io
from pathlib import Path
_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(_ROOT))
try:
    sys.stdout.reconfigure(encoding='utf-8', errors='replace')
except Exception:
    pass

import pandas as pd, numpy as np
from src import config as C
C.ensure_dirs()
print(C.describe())

KONFIGURASI PROYEK
  Repo          : C:\Users\HAJIWAHYU\Downloads\lda-topik-skripsi-si-uinrf-main
  Dokumen akhir : 285 (2021, 2022, 2023, 2024, 2025)
  Vocabulary    : 1849
  Model LDA     : K=8, alpha=asymmetric, eta=auto, passes=30, iterations=500, seed=42
  Inferensi     : Online Variational Bayes (bukan Gibbs Sampling)
  C_V resmi     : 0.4256
  Cutoff data   : 19 September 2026

## Langkah 1 - Analisis Kelengkapan Field

In [2]:
raw_df = pd.read_csv(C.SCRAPED_CSV)
print(f'Dokumen : {len(raw_df)}')
print(f'Kolom   : {list(raw_df.columns)}')
print()
for col in ['id', 'year', 'author', 'title', 'abstract']:
    filled = raw_df[col].notna().sum()
    print(f'  {col:10} terisi {filled:3d}/{len(raw_df)}  ({filled/len(raw_df)*100:5.1f}%)')

Dokumen : 312
Kolom   : ['id', 'year', 'author', 'title', 'abstract', 'url']

  id         terisi 312/312  (100.0%)
  year       terisi 312/312  (100.0%)
  author     terisi   0/312  (  0.0%)
  title      terisi 312/312  (100.0%)
  abstract   terisi   0/312  (  0.0%)

## Langkah 2 - Deteksi Duplikasi Judul

In [3]:
raw_df['tmp_title_lower'] = raw_df['title'].astype(str).str.strip().str.lower()
dupes = raw_df[raw_df.duplicated(subset=['tmp_title_lower'], keep='first')]
print(f'Dokumen dengan judul duplikat: {len(dupes)}')
if len(dupes):
    for _, r in dupes.head(5).iterrows():
        print(f'  ID {r["id"]:>7} ({r["year"]}): {str(r["title"])[:65]}')

Dokumen dengan judul duplikat: 5
  ID   41627 (2024): SISTEM INFORMASI E-LEGALISIR PADA UIN RADEN FATAH PALEMBANG DENGA
  ID   43579 (2024): ANALISIS KUALITAS WEBSITE SEKOLAH ISLAM TERPADU HARAPAN MULIA PAL
  ID   48254 (2025): Analisis User Experience Website Silayak Uin Raden Fatah Palemban
  ID   49710 (2025): ANALISIS KEPUASAN PENGGUNA APLIKASI E-DOSEN UIN RADEN FATAH PALEM
  ID   48944 (2025): ANALISIS SENTIMEN APLIKASI LIVIN BY MANDIRI PADA ULASAN PENGGUNA 

## Langkah 3 - Filter Awal dan Lineage

In [4]:
from src import data_lineage as dl

dl.run_data_lineage(verbose=True)

DATA LINEAGE & EXCLUDED DOCUMENTS TRACER

LINEAGE PER TAHUN
 tahun  raw_scraped_count  processed_count  preprocessed_final_count  dropped_raw_to_processed  dropped_preprocessed_dedup_and_length  total_dropped
  2021                 31               31                        29                         0                                      2              2
  2022                 90               87                        78                         3                                      9             12
  2023                 48               48                        46                         0                                      2              2
  2024                 51               51                        47                         0                                      4              4
  2025                 92               92                        85                         0                                      7              7

[OK] -> docs\data_lineage_by_year.csv

DOKUME

## Ringkasan Fase 02 - Data Understanding

| Tahap | Jumlah Dokumen |
|:---|---:|
| Hasil web scraping | 312 |
| Setelah filter awal | 309 |
| **Korpus bersih akhir** | **285** |

**Temuan utama:**
1. Sebagian dokumen tidak memiliki teks abstrak sehingga tidak dapat dianalisis.
2. Duplikasi judul diselesaikan dengan case-insensitive matching.
3. Setelah preprocessing, 285 dokumen lolos dengan minimal 5 token per dokumen.

## Peta Artefak

| Artefak | Dibuat oleh | Consumed by |
|:---|:---|:---|
| `data/processed/thesis_for_pipeline.csv` | Notebook 02 | Notebook 03 |
| `docs/data_lineage_by_year.csv` | Notebook 02 | Lampiran Bab III |
| `docs/excluded_documents.csv` | Notebook 02 | Lampiran Bab III |